# 📰 Financial NLP — News Sentiment & Market Movement Prediction

**Datasets**: Daily News for Stock Market Prediction + NVDA/AAPL/MSFT historical prices

## Pipeline
1. Financial news sentiment analysis (FinBERT)
2. Feature fusion: sentiment score + technical indicators (RSI, MACD, Bollinger)
3. Multivariate LSTM: next-day direction prediction
4. Temporal Fusion Transformer for time series
5. Simple signal-based backtest


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import warnings; warnings.filterwarnings('ignore')
print('Ready ✓')

In [ ]:
news = pd.read_csv('/kaggle/input/stocknews/Combined_News_DJIA.csv')
print(news.shape)
news.head(3)

## 2. FinBERT — Financial Sentiment

FinBERT is a BERT model pre-trained on financial texts (annual reports, Bloomberg news)

In [ ]:
# Load FinBERT (ProsusAI/finbert)
# from transformers import pipeline
# finbert = pipeline('text-classification', model='ProsusAI/finbert')

# Apply to headlines
# def get_sentiment_score(texts):
#     results = finbert(texts, truncation=True, max_length=512)
#     return pd.DataFrame(results)

# Aggregate by day: mean score + dominant polarity
print('FinBERT sentiment pipeline — TODO')

## 3. Technical Indicators

In [ ]:
def compute_rsi(series, window=14):
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = (-delta.clip(upper=0)).rolling(window).mean()
    return 100 - (100 / (1 + gain / loss))

def compute_macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast).mean()
    ema_slow = series.ewm(span=slow).mean()
    macd = ema_fast - ema_slow
    return macd, macd.ewm(span=signal).mean()

# TODO: load price data, compute RSI/MACD/Bollinger, merge with sentiment
print('Technical indicators defined ✓')

## 4. Multivariate LSTM — Direction Prediction

Features: [close_price_t-n...t, RSI_t, MACD_t, sentiment_score_t] → direction_{t+1}

In [ ]:
class FinancialLSTM(torch.nn.Module):
    def __init__(self, input_dim, hidden=128, layers=2, dropout=0.3):
        super().__init__()
        self.lstm = torch.nn.LSTM(input_dim, hidden, layers, batch_first=True, dropout=dropout)
        self.head = torch.nn.Sequential(
            torch.nn.Linear(hidden, 64), torch.nn.ReLU(),
            torch.nn.Dropout(0.2), torch.nn.Linear(64, 2)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

print('LSTM architecture ready ✓')